# PaddleOCR Fine-Tuning — Egyptian License Plates
Fine-tunes the Arabic PP-OCRv3 recognition model on the Egyptian license plate dataset.

## 1. Install PaddlePaddle

In [2]:

try:
    import paddle
    print(f"PaddlePaddle already installed: {paddle.__version__}")
except ModuleNotFoundError:
    import subprocess, sys
    # Pin cuda-python<13.0 before installing paddle to prevent the cu130 index
    # from upgrading it and breaking Kaggle's RAPIDS stack.
    subprocess.run([
        sys.executable, "-m", "pip", "install", "cuda-python<13.0", "-q"
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "paddlepaddle-gpu==3.3.1",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu130/",
        "-q"
    ], check=True)
    import paddle
    print(f"PaddlePaddle installed: {paddle.__version__}")

print(f"CUDA available: {paddle.is_compiled_with_cuda()}")


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


PaddlePaddle already installed: 3.3.1
CUDA available: True


## 2. Clone PaddleOCR

In [3]:
# Clone release/2.9 — compatible with PaddlePaddle 3.0+.
!rm -rf /kaggle/working/PaddleOCR
!git clone https://github.com/PaddlePaddle/PaddleOCR.git /kaggle/working/PaddleOCR -b release/2.9 -q

# PyMuPDF (PDF processing) fails to build from source on Kaggle — not needed for training.
!grep -v -i "pymupdf\|fitz" /kaggle/working/PaddleOCR/requirements.txt > /kaggle/working/requirements_filtered.txt
!pip install -r /kaggle/working/requirements_filtered.txt -q
# Note: do NOT pin numpy<2.0 — PaddlePaddle 3.3.1 supports numpy 2.x and downgrading
# breaks many pre-installed Kaggle packages (jaxlib, cupy, shap, rasterio, etc.).

## 3. Download Pretrained Arabic PP-OCRv3 Model

In [ ]:
import os
os.makedirs('/kaggle/working/pretrain_models', exist_ok=True)

!wget -q https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/arabic_PP-OCRv3_rec_train.tar \
     -O /kaggle/working/pretrain_models/arabic_PP-OCRv3_rec_train.tar

!tar -xf /kaggle/working/pretrain_models/arabic_PP-OCRv3_rec_train.tar \
     -C /kaggle/working/pretrain_models/

print('Pretrained model ready.')

## 4. Verify Dataset Paths

In [ ]:
import os

# Update this if your dataset is mounted at a different path
DATASET = '/kaggle/input/datasets/nourmhelmy/egyptian-license-plates'

checks = [
    f'{DATASET}/train_labels.txt',
    f'{DATASET}/valid_labels.txt',
    f'{DATASET}/dict.txt',
    f'{DATASET}/train',
    f'{DATASET}/valid',
]

for path in checks:
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'[{status}] {path}')

print()
print('Sample labels (raw):')
with open(f'{DATASET}/train_labels.txt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(' ', repr(line.strip()))
        if i >= 4:
            break

In [ ]:
# This cell:
# 1. Strips Windows line endings (\r\n -> \n)
# 2. Skips any malformed lines that don't have a tab
# 3. Rewrites paths so they are just 'train/filename.jpg' or 'valid/filename.jpg'
#    which is what PaddleOCR expects when joined with data_dir
# 4. Saves the fixed files to /kaggle/working/ (writable location)

import os

for split in ['train', 'valid']:
    src = f'{DATASET}/{split}_labels.txt'
    dst = f'/kaggle/working/{split}_labels.txt'

    with open(src, encoding='utf-8') as f:
        lines = f.readlines()

    fixed = []
    skipped = 0
    for line in lines:
        line = line.strip()           # removes \r\n and \n
        if '\t' not in line:
            skipped += 1
            continue
        path, text = line.split('\t', 1)
        filename = os.path.basename(path)   # keep only the image filename
        fixed.append(f'{split}/{filename}\t{text}\n')

    with open(dst, 'w', encoding='utf-8') as f:
        f.writelines(fixed)

    print(f'✅ {split}: {len(fixed)} lines written to {dst} ({skipped} skipped)')

print()
print('Sample fixed labels:')
with open('/kaggle/working/train_labels.txt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(' ', repr(line.strip()))
        if i >= 4:
            break

## 5. Train

In [4]:
# Patch rec_sar_head.py: PaddlePaddle 3.3.1 no longer allows paddle.minimum()
# on mixed int32/int64 tensors. Cast both sides to int64.
sar_head_path = '/kaggle/working/PaddleOCR/ppocr/modeling/heads/rec_sar_head.py'
with open(sar_head_path, 'r') as f:
    src = f.read()

old = (
    '                valid_width = paddle.minimum(\n'
    '                    w, paddle.ceil(valid_ratios[i] * w).astype("int32")\n'
    '                )'
)
new = (
    '                valid_width = paddle.minimum(\n'
    '                    paddle.cast(w, "int64"), paddle.cast(paddle.ceil(valid_ratios[i] * w), "int64")\n'
    '                )'
)

if old in src:
    with open(sar_head_path, 'w') as f:
        f.write(src.replace(old, new))
    print('Patched rec_sar_head.py successfully')
else:
    print('Pattern not found — already patched or structure changed')


Patched rec_sar_head.py successfully


In [5]:
# Resuming from iter_epoch_205 checkpoint.
# Global.checkpoints resumes training state (epoch, optimizer, weights).
# Global.pretrained_model is removed — checkpoints takes precedence.
!cd /kaggle/working/PaddleOCR && \
  python tools/train.py \
    -c configs/rec/PP-OCRv3/multi_language/arabic_PP-OCRv3_rec.yml \
    -o Global.checkpoints=/kaggle/working/output/rec_arabic_plates/iter_epoch_205 \
       Global.character_dict_path=/kaggle/input/datasets/nourmhelmy/egyptian-license-plates/dict.txt \
       Global.max_text_length=15 \
       Global.epoch_num=300 \
       Global.save_model_dir=/kaggle/working/output/rec_arabic_plates \
       Global.save_epoch_step=5 \
       Global.eval_batch_step="[30,30]" \
       Global.use_visualdl=False \
       Global.save_res_path=/kaggle/working/output/predicts.txt \
       Train.dataset.data_dir=/kaggle/input/datasets/nourmhelmy/egyptian-license-plates \
       "Train.dataset.label_file_list=['/kaggle/working/train_labels.txt']" \
       Train.loader.num_workers=0 \
       Train.loader.batch_size_per_card=32 \
       Eval.dataset.data_dir=/kaggle/input/datasets/nourmhelmy/egyptian-license-plates \
       "Eval.dataset.label_file_list=['/kaggle/working/valid_labels.txt']" \
       Eval.loader.num_workers=0 \
  2>&1 | tee /kaggle/working/train.log


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2026/04/02 17:08:48] ppocr INFO: Architecture : 
[2026/04/02 17:08:48] ppocr INFO:     Backbone : 
[2026/04/02 17:08:48] ppocr INFO:         last_conv_stride : [1, 2]
[2026/04/02 17:08:48] ppocr INFO:         last_pool_kernel_size : [2, 2]
[2026/04/02 17:08:48] ppocr INFO:         last_pool_type : avg
[2026/04/02 17:08:48] ppocr INFO:         name : MobileNetV1Enhance
[2026/04/02 17:08:48] ppocr INFO:         scale : 0.5
[2026/04/02 17:08:48] ppocr INFO:     Head : 
[2026/04/02 17:08:48] ppocr INFO:         head_list : 
[2026/04/02 17:08:48] ppocr INFO:             CTCHead : 
[2026/04/02 17:08:48] ppocr INFO:                 Head : 
[2026/04/02 17:08:48] ppocr INFO:  

In [ ]:
import os

DATASET = '/kaggle/input/datasets/nourmhelmy/egyptian-license-plates'

for split in ['train', 'valid']:
    label_file = f'/kaggle/working/{split}_labels.txt'
    missing = []
    
    with open(label_file, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if '\t' not in line:
                continue
            path = line.split('\t')[0]
            full_path = os.path.join(DATASET, path)
            if not os.path.exists(full_path):
                missing.append(full_path)
    
    print(f'{split}: {len(missing)} missing images')
    for p in missing[:5]:  # show first 5
        print(f'  MISSING: {p}')

In [ ]:
import os
from PIL import Image

DATASET = '/kaggle/input/datasets/nourmhelmy/egyptian-license-plates'

for split in ['train', 'valid']:
    label_file = f'/kaggle/working/{split}_labels.txt'
    corrupted = []

    with open(label_file, encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if '\t' not in line:
            continue
        path, text = line.split('\t', 1)
        full_path = os.path.join(DATASET, path)
        try:
            with Image.open(full_path) as img:
                img.verify()
        except Exception as e:
            corrupted.append((full_path, str(e)))

    print(f'{split}: {len(lines)} total, {len(corrupted)} corrupted')
    for path, err in corrupted:
        print(f'  CORRUPTED: {path} — {err}')

In [ ]:
DATASET = '/kaggle/input/datasets/nourmhelmy/egyptian-license-plates'

# Load dictionary
with open(f'{DATASET}/dict.txt', encoding='utf-8') as f:
    dict_chars = set(c.strip() for c in f.readlines())

print(f'Dictionary has {len(dict_chars)} characters: {sorted(dict_chars)}')
print()

# Check every label
for split in ['train', 'valid']:
    unknown = set()
    with open(f'/kaggle/working/{split}_labels.txt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if '\t' not in line:
                continue
            text = line.split('\t', 1)[1]
            for char in text:
                if char not in dict_chars:
                    unknown.add(char)
    
    if unknown:
        print(f'{split}: ⚠️ {len(unknown)} characters in labels NOT in dictionary:')
        for c in sorted(unknown):
            print(f'  repr: {repr(c)}  unicode: U+{ord(c):04X}')
    else:
        print(f'{split}: ✅ all characters are in the dictionary')

## 6. Export Best Model for Inference

In [6]:
!cd /kaggle/working/PaddleOCR && \
  python tools/export_model.py \
    -c configs/rec/PP-OCRv3/multi_language/arabic_PP-OCRv3_rec.yml \
    -o Global.pretrained_model=/kaggle/working/output/rec_arabic_plates/best_accuracy \
       Global.character_dict_path=/kaggle/input/datasets/nourmhelmy/egyptian-license-plates/dict.txt \
       Global.max_text_length=15 \
       Global.save_inference_dir=/kaggle/working/inference/arabic_plates_rec

print('Inference model saved to /kaggle/working/inference/arabic_plates_rec')

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
W0402 19:53:07.651327  2079 gpu_resources.cc:116] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 13.0
[2026/04/02 19:53:08] ppocr INFO: load pretrain successful from /kaggle/working/output/rec_arabic_plates/best_accuracy
W0402 19:53:09.264142  2079 eager_utils.cc:3584] Paddle static graph(PIR) not support input out tensor for now!!!!!
[2026/04/02 19:53:09] ppocr INFO: inference model is saved to /kaggle/working/inference/arabic_plates_rec/inference
[2026/04/02 19:53:09] ppocr INFO: Export inference config file to /kaggle/working/inference/arabic_plates_rec/inference.yml
Inference model saved to /kaggle/working/infere

## 7. Quick Test on Validation Set

In [7]:
!cd /kaggle/working/PaddleOCR && \
  python tools/eval.py \
    -c configs/rec/PP-OCRv3/multi_language/arabic_PP-OCRv3_rec.yml \
    -o Global.pretrained_model=/kaggle/working/output/rec_arabic_plates/best_accuracy \
       Global.character_dict_path=/kaggle/input/datasets/nourmhelmy/egyptian-license-plates/dict.txt \
       Global.max_text_length=15 \
       Eval.dataset.data_dir=/kaggle/input/datasets/nourmhelmy/egyptian-license-plates \
       "Eval.dataset.label_file_list=['/kaggle/working/valid_labels.txt']" \
       Eval.loader.num_workers=0

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2026/04/02 20:03:47] ppocr INFO: Architecture : 
[2026/04/02 20:03:47] ppocr INFO:     Backbone : 
[2026/04/02 20:03:47] ppocr INFO:         last_conv_stride : [1, 2]
[2026/04/02 20:03:47] ppocr INFO:         last_pool_kernel_size : [2, 2]
[2026/04/02 20:03:47] ppocr INFO:         last_pool_type : avg
[2026/04/02 20:03:47] ppocr INFO:         name : MobileNetV1Enhance
[2026/04/02 20:03:47] ppocr INFO:         scale : 0.5
[2026/04/02 20:03:47] ppocr INFO:     Head : 
[2026/04/02 20:03:47] ppocr INFO:         head_list : 
[2026/04/02 20:03:47] ppocr INFO:             CTCHead : 
[2026/04/02 20:03:47] ppocr INFO:                 Head : 
[2026/04/02 20:03:47] ppocr INFO:  